In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import mplscience
import seaborn as sns
import cellrank as cr
import scanpy as sc
import scvelo as scv
from anndata import AnnData
import pickle
import resource
import unitvelo as utv
from cr2 import running_in_notebook
import time
import dynamo as dyn
import warnings
import numpy as np
import os
warnings.simplefilter("ignore", category=UserWarning)
sys.path.extend(["../../../", "."])
# from paths import DATA_DIR, FIG_DIR  
# set verbosity levels
sc.settings.verbosity = 2
cr.settings.verbosity = 4
scv.settings.verbosity = 3
SAVE_FIGURES = False
FIGURE_FORMAT = "pdf"
N_JOBS = 8
method='cellrank'

In [ ]:
#dataset name
datasets=['Pancreas','DentateGyrus','Erythroid_Maturation','HumanBoneMarrow','Intestinal_organoid','mouse_retina','Hindbrain_GABA_Glio','organogenesis_chondrocyte']
#data path
data_dir = '/data/'
#result path
save_dir = '/result/'

In [ ]:
df_CB= pd.DataFrame(columns=['Mean', 'Time(s)'])
print(df_CB)
df_IC= pd.DataFrame(columns=['Mean', 'Time(s)'])
print(df_IC)

In [ ]:
for dataset in datasets:
    print(dataset)
    adata = sc.read_h5ad(data_dir + dataset +'/'+ f'{dataset}.h5ad')
    scv.pp.filter_and_normalize(adata, min_counts=20, n_top_genes=3000)
    scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
    
    start = time.time()
    memory_before = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    scv.tl.recover_dynamics(adata, n_jobs=N_JOBS)
    scv.tl.velocity(adata,vkey='velocity',mode="dynamical")
    vk = cr.kernels.VelocityKernel(adata).compute_transition_matrix()
    ck = cr.kernels.ConnectivityKernel(adata).compute_transition_matrix()
    combined_kernel = 0.8 * vk + 0.2 * ck
    adata.uns['velocity_graph']=combined_kernel.transition_matrix
    end = time.time()
    title=f'{dataset}_{method}'
    fig, ax = plt.subplots(1, 1, figsize = (8, 6))
    vk.plot_projection(ax = ax,title=title,save=save_dir + method +'/'+'CB_IC/'+ f'{dataset}_{method}.svg')

    # Calculate performance metrics:
    file = open(data_dir + dataset +'/'+f'{dataset}_groundTruth.pickle' ,'rb')
    ground_truth = pickle.load(file)
    dyn.tl.neighbors(adata,n_neighbors=30)
    adata.obsm['velocity_umap']=adata.obsm['T_fwd_umap']
    metrics = utv.evaluate(adata, ground_truth, 'clusters', 'velocity')
    if os.path.exists(save_dir + method + '/' + '_CBDir_scores.csv'):
        tab = pd.read_csv(save_dir + method +'/'+'_CBDir_scores.csv', index_col = 0)
    else:
        tab_cb = pd.DataFrame(columns = list(metrics['Cross-Boundary Direction Correctness (A->B)'].keys())  + ['Mean', 'Time(s)'],index = [dataset])
        tab_IC = pd.DataFrame(columns = list(metrics['In-cluster Coherence'].keys())  + ['Mean', 'Time(s)'],index = [dataset])
    ##CBDC_scores
    cb_score = [np.mean(metrics['Cross-Boundary Direction Correctness (A->B)'][x])
                for x in metrics['Cross-Boundary Direction Correctness (A->B)'].keys()]
    tab_cb.loc[dataset,:] = cb_score + [np.mean(cb_score), end-start]
    df_CB=df_CB.append(pd.DataFrame([[np.mean(cb_score), end-start]],columns=df_CB.columns,index=[dataset]))
    tab_cb.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_CBDir_scores.csv')
    ##ICCoh_scores
    IC_score = [np.mean(metrics['In-cluster Coherence'][x])
                for x in metrics['In-cluster Coherence'].keys()]
    tab_IC.loc[dataset,:] = IC_score + [np.mean(IC_score), end-start]
    df_IC=df_IC.append(pd.DataFrame([[np.mean(IC_score), end-start]],columns=df_IC.columns,index=[dataset]))
    tab_IC.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_ICCoh_scores.csv')
    adata.write_h5ad(save_dir + method +'/' +'CB_IC/'+ f'{dataset}_AnnData_Forscore.h5ad')

In [ ]:
df_CB.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{method}_CBDir_scores.csv')
df_IC.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{method}_ICCoh_scores.csv')